In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# ==========================================
# 1. SETUP 
# ==========================================

try:
    # Load the Stats File
    df_stats = pd.read_excel('football_data.xlsx')
    
    # Clean up text
    df_stats.columns = df_stats.columns.str.strip()
    df_stats['Team'] = df_stats['Team'].astype(str).str.strip()

    # Calculate League Averages
    league_avgs = df_stats.groupby('League')[['Home_xG', 'Home_xGA', 'Away_xG', 'Away_xGA']].mean()
    print(f"Successfully loaded stats for {len(df_stats)} teams from football_data.xlsx.")

except FileNotFoundError:
    print("Error: Could not find 'football_data.xlsx'.")
    exit()
except Exception as e:
    print(f"Error reading Excel file: {e}")
    exit()

def get_match_prediction(home_team, away_team):
    try:
        # 1. GET STATS
        h_stats = df_stats[df_stats['Team'] == home_team].iloc[0]
        a_stats = df_stats[df_stats['Team'] == away_team].iloc[0]
        l_avg = league_avgs.loc[h_stats['League']]
        
        # 2. CALCULATE EFFICIENCY 
        # Formula: Season Goals / Season xG
        # We handle potential divide-by-zero by defaulting to 1.0
        if h_stats['Season_xG'] > 0:
            h_eff = h_stats['Season_Goals'] / h_stats['Season_xG']
        else:
            h_eff = 1.0
            
        if a_stats['Season_xG'] > 0:
            a_eff = a_stats['Season_Goals'] / a_stats['Season_xG']
        else:
            a_eff = 1.0
            
        # Keep efficiency between 0.85 (-15%) and 1.15 (+15%)
        # This prevents a lucky team from breaking the model
        h_eff = max(0.85, min(h_eff, 1.15))
        a_eff = max(0.85, min(a_eff, 1.15))
        
        # 3. CALCULATE RAW EXPECTED GOALS (The Engine)
        raw_exp_h = (h_stats['Home_xG']/l_avg['Home_xG']) * (a_stats['Away_xGA']/l_avg['Home_xG']) * l_avg['Home_xG']
        raw_exp_a = (a_stats['Away_xG']/l_avg['Away_xG']) * (h_stats['Home_xGA']/l_avg['Away_xG']) * l_avg['Away_xG']
        
        # 4. APPLY EFFICIENCY ADJUSTMENT
        exp_h = raw_exp_h * h_eff
        exp_a = raw_exp_a * a_eff
        
        # 5. POISSON PROBABILITIES
        max_g = 10
        grid = np.outer(poisson.pmf(np.arange(max_g), exp_h), poisson.pmf(np.arange(max_g), exp_a))
        
        p_away = np.sum(np.triu(grid, 1))
        p_draw = np.sum(np.diag(grid))
        p_home = np.sum(np.tril(grid, -1))
        
        print(f"\n--- {home_team} vs {away_team} ---")
        print(f"Finishing Adj: {home_team} ({h_eff:.2f}x) | {away_team} ({a_eff:.2f}x)")
        print(f"Probabilities: Home {p_home:.1%} | Draw {p_draw:.1%} | Away {p_away:.1%}")
        
        # 6. LOGIC
        
        # A. Winner Pick
        if p_home > 0.50:
            print(f" Pick: {home_team} to Win (Strong Confidence)")
        elif p_away > 0.50:
            print(f" Pick: {away_team} to Win (Strong Confidence)")
            
        # B. Draw Detector 
        elif p_draw > 0.26:
            print(f" ALERT: High Draw Chance ({p_draw:.1%}).")
        
        # C. Value Trap
        else:
            print(f" Pick: {home_team if p_home > p_away else away_team} (Low Confidence - Risky)")

    except IndexError:
        print(f"Error: Could not find stats for '{home_team}' or '{away_team}'. Check spelling!")
    except KeyError as e:
        print(f"Error: Data missing for one of the teams. Details: {e}")

# ==========================================
# 3. ENTER GAMES HERE
# ==========================================

get_match_prediction("Arsenal", "Chelsea")
get_match_prediction("Fiorentina", "Pisa")

Successfully loaded stats for 96 teams from football_data.xlsx.

--- Arsenal vs Chelsea ---
Finishing Adj: Arsenal (0.99x) | Chelsea (0.85x)
Probabilities: Home 57.4% | Draw 23.4% | Away 19.3%
 Pick: Arsenal to Win (Strong Confidence)

--- Fiorentina vs Pisa ---
Finishing Adj: Fiorentina (0.85x) | Pisa (0.85x)
Probabilities: Home 57.9% | Draw 20.5% | Away 21.7%
 Pick: Fiorentina to Win (Strong Confidence)


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# Additional model to find expected scoreline + xG.
# ==========================================
# 1. SETUP 
# ==========================================
try:
    # Load the Stats File
    df_stats = pd.read_excel('football_data.xlsx') 
    
    # Clean up text
    df_stats.columns = df_stats.columns.str.strip()
    df_stats['Team'] = df_stats['Team'].astype(str).str.strip()
    
    # Calculate League Averages
    league_avgs = df_stats.groupby('League')[['Home_xG', 'Home_xGA', 'Away_xG', 'Away_xGA']].mean()
    print(f"Successfully loaded stats for {len(df_stats)} teams from football_data.xlsx.")
    
except FileNotFoundError:
    print("Error: Could not find 'football_data.xlsx'.")
    exit()
except Exception as e:
    print(f"Error reading Excel file: {e}")
    exit()

def get_match_prediction(home_team, away_team):
    try:
        # 1. GET STATS
        h_stats = df_stats[df_stats['Team'] == home_team].iloc[0]
        a_stats = df_stats[df_stats['Team'] == away_team].iloc[0]
        l_avg = league_avgs.loc[h_stats['League']]
        
        # 2. CALCULATE EFFICIENCY (The New "Triple Crown" Feature)
        if h_stats['Season_xG'] > 0:
            h_eff = h_stats['Season_Goals'] / h_stats['Season_xG']
        else:
            h_eff = 1.0
            
        if a_stats['Season_xG'] > 0:
            a_eff = a_stats['Season_Goals'] / a_stats['Season_xG']
        else:
            a_eff = 1.0
            
        # CLAMP: Keep efficiency between 0.85 (-15%) and 1.15 (+15%)
        h_eff = max(0.85, min(h_eff, 1.15))
        a_eff = max(0.85, min(a_eff, 1.15))
        
        # 3. CALCULATE RAW EXPECTED GOALS (The Engine)
        raw_exp_h = (h_stats['Home_xG']/l_avg['Home_xG']) * (a_stats['Away_xGA']/l_avg['Home_xG']) * l_avg['Home_xG']
        raw_exp_a = (a_stats['Away_xG']/l_avg['Away_xG']) * (h_stats['Home_xGA']/l_avg['Away_xG']) * l_avg['Away_xG']
        
        # 4. APPLY EFFICIENCY ADJUSTMENT
        exp_h = raw_exp_h * h_eff
        exp_a = raw_exp_a * a_eff
        
        # 5. POISSON PROBABILITIES
        max_g = 10
        grid = np.outer(poisson.pmf(np.arange(max_g), exp_h), poisson.pmf(np.arange(max_g), exp_a))
        
        p_away = np.sum(np.triu(grid, 1))
        p_draw = np.sum(np.diag(grid))
        p_home = np.sum(np.tril(grid, -1))

        # --- FIND MOST LIKELY SCORELINE ---
        # np.argmax finds the highest value in the 1D flattened array
        # np.unravel_index converts that back into the 2D grid coordinates (Home Goals, Away Goals)
        most_likely_idx = np.unravel_index(np.argmax(grid), grid.shape)
        pred_h_goals, pred_a_goals = most_likely_idx
        score_prob = grid[pred_h_goals, pred_a_goals]
        
        # --- PRINT BLOCK ---
        print(f"\n--- {home_team} vs {away_team} ---")
        print(f"Finishing Adj: {home_team} ({h_eff:.2f}x) | {away_team} ({a_eff:.2f}x)")
        print(f"Expected Goals (xG): {home_team} {exp_h:.2f} | {away_team} {exp_a:.2f}")
        print(f"Most Likely Score: {home_team} {pred_h_goals} - {pred_a_goals} {away_team} ({score_prob:.1%} chance)")
        print(f"Probabilities: Home {p_home:.1%} | Draw {p_draw:.1%} | Away {p_away:.1%}")
        
        # 6. LOGIC
        
        # A. Winner Pick
        if p_home > 0.50:
            print(f" Pick: {home_team} to Win (Strong Confidence)")
        elif p_away > 0.50:
            print(f" Pick: {away_team} to Win (Strong Confidence)")
            
        # B. Draw Detector
        elif p_draw > 0.26:
            print(f" ALERT: High Draw Chance ({p_draw:.1%}). Consider betting DRAW or Double Chance.")
        
        # C. Value Trap
        else:
            print(f" Pick: {home_team if p_home > p_away else away_team} (Low Confidence - Risky)")

    except IndexError:
        print(f"Error: Could not find stats for '{home_team}' or '{away_team}'. Check spelling!")
    except KeyError as e:
        print(f"Error: Data missing for one of the teams. Details: {e}")

# ==========================================
# 3. ENTER TOMORROW'S GAMES HERE
# ==========================================

get_match_prediction("Fulham", "Tottenham")
get_match_prediction("Brighton", "Nottingham")

Successfully loaded stats for 96 teams from football_data.xlsx.

--- Fulham vs Tottenham ---
Finishing Adj: Fulham (1.13x) | Tottenham (1.15x)
Expected Goals (xG): Fulham 1.39 | Tottenham 1.31
Most Likely Score: Fulham 1 - 1 Tottenham (12.2% chance)
Probabilities: Home 38.8% | Draw 25.8% | Away 35.4%
 Pick: Fulham (Low Confidence - Risky)

--- Brighton vs Nottingham ---
Finishing Adj: Brighton (0.90x) | Nottingham (0.85x)
Expected Goals (xG): Brighton 1.46 | Nottingham 0.79
Most Likely Score: Brighton 1 - 0 Nottingham (15.4% chance)
Probabilities: Home 53.2% | Draw 26.7% | Away 20.2%
 Pick: Brighton to Win (Strong Confidence)


In [4]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from sklearn.preprocessing import label_binarize
import os

# ==========================================
# 1. SETUP & LOAD
# ==========================================
FILENAME = 'matches.xlsx'

try:
    print(f"Loading {FILENAME}...")
    # Add a fallback just in case you are using the CSV version
    if os.path.exists(FILENAME):
        df = pd.read_excel(FILENAME)
    elif os.path.exists('matches.csv'):
        df = pd.read_csv('matches.csv')
    else:
        raise FileNotFoundError
        
    df.columns = df.columns.str.strip()
    
    # --- CLEANING STEP ---
    # 1. Remove invisible spaces from team names
    df['HomeTeam'] = df['HomeTeam'].astype(str).str.strip()
    df['AwayTeam'] = df['AwayTeam'].astype(str).str.strip()
    
    # 2. Fix known typos
    typo_fix = {
        'Totttenham': 'Tottenham',
        'West Han': 'West Ham',
        'Man Utd': 'Man United', 
        'Manchester United': 'Man United'
    }
    df['HomeTeam'] = df['HomeTeam'].replace(typo_fix)
    df['AwayTeam'] = df['AwayTeam'].replace(typo_fix)
    # ---------------------------

    # Required columns adapted for matches.xlsx
    required = ['Date', 'League', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Home_xG', 'Away_xG']
    if not all(col in df.columns for col in required):
        print(f"❌ Error: Missing columns. Needed: {required}")
        exit()

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Drop rows that don't have results yet (future games)
    df = df.dropna(subset=['FTHG', 'FTAG'])
    print(f"✅ Loaded {len(df)} matches. Running Efficiency Test...")

except FileNotFoundError:
    print(f"❌ Error: '{FILENAME}' not found.")
    exit()

# ==========================================
# 2. THE ENGINE 
# ==========================================
predictions = []
actuals = []
leagues_tracked = [] 
min_games = 6

for i in range(len(df)):
    # A. Current Match
    row = df.iloc[i]
    date, home, away, league = row['Date'], row['HomeTeam'], row['AwayTeam'], row['League']

    # B. History (Strictly Past)
    history = df[df['Date'] < date]
    league_hist = history[history['League'] == league]
    
    if len(league_hist) < 20: continue 

    # C. Calculate Baselines
    avg_h_xg = league_hist['Home_xG'].mean()
    avg_a_xg = league_hist['Away_xG'].mean()
    
    # Specific Home/Away records
    h_home_rec = league_hist[league_hist['HomeTeam'] == home]
    a_away_rec = league_hist[league_hist['AwayTeam'] == away]
    
    if len(h_home_rec) < min_games or len(a_away_rec) < min_games:
        continue

    # --- 1. CALCULATE xG STRENGTH ---
    h_att = h_home_rec['Home_xG'].mean() / avg_h_xg
    h_def = h_home_rec['Away_xG'].mean() / avg_a_xg # Away_xG here is what Home conceded
    
    a_att = a_away_rec['Away_xG'].mean() / avg_a_xg
    a_def = a_away_rec['Home_xG'].mean() / avg_h_xg # Home_xG here is what Away conceded
    
    raw_exp_h = h_att * a_def * avg_h_xg
    raw_exp_a = a_att * h_def * avg_a_xg

    # --- 2. CALCULATE EFFICIENCY ---
    # Home Team Efficiency
    h_goals = h_home_rec['FTHG'].sum()
    h_xg_total = h_home_rec['Home_xG'].sum()
    h_eff = h_goals / h_xg_total if h_xg_total > 0 else 1.0
    
    # Away Team Efficiency
    a_goals = a_away_rec['FTAG'].sum()
    a_xg_total = a_away_rec['Away_xG'].sum()
    a_eff = a_goals / a_xg_total if a_xg_total > 0 else 1.0
    
    # SAFETY CLAMP
    h_eff = max(0.85, min(h_eff, 1.15))
    a_eff = max(0.85, min(a_eff, 1.15))

    # --- 3. APPLY EFFICIENCY ---
    final_exp_h = raw_exp_h * h_eff
    final_exp_a = raw_exp_a * a_eff
    
    # --- 4. PREDICT ---
    max_g = 10
    grid = np.outer(poisson.pmf(np.arange(max_g), final_exp_h), 
                    poisson.pmf(np.arange(max_g), final_exp_a))
    
    p_away = np.sum(np.triu(grid, 1))
    p_draw = np.sum(np.diag(grid))
    p_home = np.sum(np.tril(grid, -1))
    
    predictions.append([p_away, p_draw, p_home])
    leagues_tracked.append(league) 
    
    # Actual Result
    if row['FTHG'] > row['FTAG']: res = 2
    elif row['FTHG'] == row['FTAG']: res = 1
    else: res = 0
    actuals.append(res)

# ==========================================
# 3. RESULTS
# ==========================================
if len(actuals) < 10:
    print("Not enough data.")
else:
    # A. OVERALL STATS
    y_test = label_binarize(actuals, classes=[0, 1, 2])
    y_pred = np.array(predictions)
    
    roc = roc_auc_score(y_test, y_pred, multi_class='ovr')
    acc = accuracy_score(actuals, np.argmax(y_pred, axis=1))
    loss = log_loss(actuals, y_pred)
    
    print(f"\n" + "="*50)
    print(f"OVERALL RESULTS ({len(actuals)} Matches)")
    print(f"="*50)
    print(f"ROC AUC Score:   {roc:.4f}")
    print(f"Accuracy:        {acc:.1%}")
    print(f"Log Loss:        {loss:.4f}")
    print(f"="*50)
    
    # B. LEAGUE BY LEAGUE BREAKDOWN
    df_results = pd.DataFrame({
        'Actual': actuals,
        'League': leagues_tracked
    })
    
    unique_leagues = df_results['League'].unique()
    
    print(f"\nLEAGUE PERFORMANCE BREAKDOWN")
    print(f"{'League':<20} | {'Matches':<8} | {'ROC':<8} | {'Acc':<8}")
    print("-" * 55)
    
    for lg in unique_leagues:
        indices = df_results.index[df_results['League'] == lg].tolist()
        if len(indices) < 5: continue 
        
        y_true_lg = np.array([actuals[i] for i in indices])
        y_pred_lg = np.array([predictions[i] for i in indices])
        
        try:
            if len(np.unique(y_true_lg)) > 1:
                y_test_lg = label_binarize(y_true_lg, classes=[0, 1, 2])
                roc_lg = roc_auc_score(y_test_lg, y_pred_lg, multi_class='ovr')
                roc_str = f"{roc_lg:.3f}"
            else:
                roc_str = "N/A"
            
            acc_lg = accuracy_score(y_true_lg, np.argmax(y_pred_lg, axis=1))
            print(f"{lg:<20} | {len(indices):<8} | {roc_str:<8} | {acc_lg:.1%}")
            
        except Exception as e:
            print(f"{lg:<20} | Error calculating stats")

Loading matches.xlsx...
✅ Loaded 1247 matches. Running Efficiency Test...

OVERALL RESULTS (762 Matches)
ROC AUC Score:   0.6210
Accuracy:        48.8%
Log Loss:        1.0209

LEAGUE PERFORMANCE BREAKDOWN
League               | Matches  | ROC      | Acc     
-------------------------------------------------------
La Liga              | 183      | 0.617    | 48.6%
Bundesliga           | 163      | 0.668    | 49.1%
Serie A              | 210      | 0.619    | 52.4%
EPL                  | 206      | 0.593    | 45.1%


C:\Users\jordi\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:259: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
